In [1]:
import os
import xarray as xr
import sys
import itertools
from itertools import zip_longest
from lca_algebraic import initProject
import pandas as pd
import numpy as np
import xarray as xr
import lca_algebraic as agb
from lca_algebraic import *
from lca_algebraic.stats import * 
import bw2data, bw2io
import bw2calc
from premise import *
import premise as prm
from scipy.integrate import cumtrapz

In [2]:
sys.path.append('/Users/susierwu/Dropbox/DLCA_DynCO2/pdpLCA_GT/') 
from utils import *
from utils.setup import *
#from utils.setup_whdLCIA import *
from utils.getAct import *
from utils.utils import *
from utils.dlcia_functions import *

modify the setup.py file for reading Parameter excel sheet. here we use 'data/p_file/p_tech_perSSP_Y.xlsx' as the technology parameter file.  
Define your chosen methods in the setup.py file. Current methods are:  [('IPCC 2021 no LT', 'climate change no LT', 'global warming potential (GWP100) no LT'), ('IPCC 2021', 'climate change', 'GWP 100a, incl. H and bio CO2')]
Mapping premise_remind_DB to SSPx:  {'SSP1-PkBudg500': 'ssp119', 'SSP2-Base': 'ssp245', 'SSP5-Base': 'ssp585'}
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
pGWP100 are imported as:  [('IPCC 2021 - dpCFsSSP119_MY2030 - year100', 'climate change', 'pGWP100'), ('IPCC 2021 - dpCFsSSP245_MY2030 - year100', 'climate change', 'pGWP100'), ('IPCC 2021 - dpCFsSSP585_MY2030 - year100', 'climate change', 'pGWP100'), ('IPCC 2021 - dpCFsSSP119_MY2040 - year100', 'climate change', 'pGWP100'), ('IPCC 2021 - dpCFsSSP245_MY2040 - year100', 'climate change', 'pGWP100'), ('IPCC 2021

In [3]:
from utils_pGWP.pGWP_F2_V1 import *

In [4]:
bw2data.projects.set_current('iveo_v1')
mybio = bw2data.Database("biosphere3")
initProject("iveo_v1")

Biosphere database already present!!! No setup is needed


In [5]:
print( ssp_scn  , ssp_yr) 
bw2_method_metric = 'pGWP'
root_path = '/Users/susierwu/Dropbox/DLCA_DynCO2/pdpLCA_GT/'
tech_p_file = os.path.abspath(os.path.join(root_path, 'data/P_files/p_tech_perSSP_Y.xlsx'))
consmp_p_file = os.path.abspath(os.path.join(root_path, 'data/P_files/tech_conversion_veh_p/p_scn_v_need_tech.xlsx'))

['_remind_SSP1-PkBudg500', '_remind_SSP2-Base', '_remind_SSP5-Base'] [2030, 2040, 2050]


In [6]:
v_lifetime = 20  
b_lifetime = 10 

### init 

In [7]:
pdb = bw.Database("iveo_Parameterized_v1")
print(len(pdb))
agb.loadParams()

384


[ParamRegistry] Param NMC622_battery_cell_density was already defined in '<project>' : overriding.
[ParamRegistry] Param NCA_battery_cell_density was already defined in '<project>' : overriding.
[ParamRegistry] Param LFP_battery_cell_density was already defined in '<project>' : overriding.
[ParamRegistry] Param LTO_battery_cell_density was already defined in '<project>' : overriding.
[ParamRegistry] Param LSB_battery_cell_density was already defined in '<project>' : overriding.
[ParamRegistry] Param LSB_cell_assmb_NG_MJperkg was already defined in '<project>' : overriding.
[ParamRegistry] Param LSB_cell_assmb_elec_kWhperkg was already defined in '<project>' : overriding.
[ParamRegistry] Param g_truck_vehicle_prod_lightweighting was already defined in '<project>' : overriding.
[ParamRegistry] Param battery_total_capacity_for_gtruck was already defined in '<project>' : overriding.
[ParamRegistry] Param biodiesel_UCO_perc was already defined in '<project>' : overriding.
[ParamRegistry] Pa

### get already created UPRs:

In [8]:
bat_finalwhbat_switch_allsspyear = [] 

for prm_name in ssp_scn:
    for year_t in ssp_yr: 
        bg_db_ssp_yr = bg_ei_ + prm_name + "_" + str(year_t)            # backgroun database (BD) to use for SSP[x], Year[t]
        copied_prm_UPR_suffix = "_from_" + prm_name + "_" + str(year_t) # copy UPR from BD to PDB for further use/modification if needed
        newUPR_suffix =  "_wh_" + prm_name + "_" + str(year_t)          # naming on any new UPRs created in PDB

        # final one unit of g_truck with lightweighting as Parameter: 
        bat_final_whbatswitch = agb.findActivity(
                db_name = PDB_NAME,
                name="V1B_bat_g-truck BEV_wh_switch_bat" + newUPR_suffix)

        bat_finalwhbat_switch_allsspyear.append(bat_final_whbatswitch)


len(bat_finalwhbat_switch_allsspyear)

9

### read in TRL table to see which switch_P to apply

In [9]:
data = xr.open_dataset( os.path.abspath(os.path.join(root_path, 'data/to_use_interm/TRL_1C_vehDSwhTRL.nc')))
data2 = data.where(data["tech_avail"] == 1, drop = True) 
#data2

### C. Calc 
C.0 read all P file 

In [10]:
for bat_final_whSwitchbat in bat_finalwhbat_switch_allsspyear:
    print(bat_final_whSwitchbat)
    # need a matched YearlyParams for each SSPx:
    str1 = bat_final_whSwitchbat.as_dict()["name"]
    str2 = str1.split("remind_")[1]
    year_to_get_inP = int( str2.split("_")[-1] )
    ssp_remind = str2.split("_")[0] 
    ssp_to_get_inP = ssp_remind_Pname_map.get(ssp_remind)

    yearlyParams = loadYearlyParams_multisheet(PARAMS_FILE = tech_p_file, s_name = ['V1A_g_truck', 'V1B_bat_LSB_perkWh', 'V1B_bat_prmLIBs_perkWh'], years = [year_to_get_inP], SSP = ssp_to_get_inP) 
    #print(f"for SSP {ssp_to_get_inP}, year {year_to_get_inP}, P:  {yearlyParams}" )


'V1B_bat_g-truck BEV_wh_switch_bat_wh__remind_SSP1-PkBudg500_2030' (bat_for_one_truck, GLO, None)
'V1B_bat_g-truck BEV_wh_switch_bat_wh__remind_SSP1-PkBudg500_2040' (bat_for_one_truck, GLO, None)
'V1B_bat_g-truck BEV_wh_switch_bat_wh__remind_SSP1-PkBudg500_2050' (bat_for_one_truck, GLO, None)
'V1B_bat_g-truck BEV_wh_switch_bat_wh__remind_SSP2-Base_2030' (bat_for_one_truck, GLO, None)
'V1B_bat_g-truck BEV_wh_switch_bat_wh__remind_SSP2-Base_2040' (bat_for_one_truck, GLO, None)
'V1B_bat_g-truck BEV_wh_switch_bat_wh__remind_SSP2-Base_2050' (bat_for_one_truck, GLO, None)
'V1B_bat_g-truck BEV_wh_switch_bat_wh__remind_SSP5-Base_2030' (bat_for_one_truck, GLO, None)
'V1B_bat_g-truck BEV_wh_switch_bat_wh__remind_SSP5-Base_2040' (bat_for_one_truck, GLO, None)
'V1B_bat_g-truck BEV_wh_switch_bat_wh__remind_SSP5-Base_2050' (bat_for_one_truck, GLO, None)


### C.1  run static pLCIA with: 
IPCC,  premise_gwp, pGWP100 

In [16]:
#GWP_list, SSP_list, Year_list, tech_list  = [], [],[], []
GWP0_list, GWP1_list, GWP2_list, SSP_list, Year_list, tech_list = [], [],[],[],[],[]

for bat_final_whSwitchbat in bat_finalwhbat_switch_allsspyear:
    str1 = bat_final_whSwitchbat.as_dict()["name"]
    str2 = str1.split("remind_")[1]
    year_to_get_inP = int( str2.split("_")[-1] )
    ssp_remind = str2.split("_")[0] 
    ssp_to_get_inP = ssp_remind_Pname_map.get(ssp_remind)
    
    """ prepare methods: GWP1_list using static GWP,  GWP2_list using the pGWP100 for the matched v_year """ 
    gwp_spx_yt = get_my_pGWP100(ssp = ssp_to_get_inP, my = year_to_get_inP, metric = 'pGWP') 
    methods_touse = [chosen_methods[0], chosen_methods[1], gwp_spx_yt] # IPCCGWP100, premiseGWP, pGWP100_per_premiseGWP

    """ see what tech can be used under SSPx, Yeart, b_replacement != v_year """
    year_to_get_inP_Yof_bat_repl = year_to_get_inP + b_lifetime
    if year_to_get_inP_Yof_bat_repl <= 2050: 
        year_to_get_inP_Yof_bat_repl2 = year_to_get_inP_Yof_bat_repl 
    else: 
        year_to_get_inP_Yof_bat_repl2 = 2050 
    
    yearlyParams = loadYearlyParams_multisheet(PARAMS_FILE = tech_p_file, s_name = ['V1A_g_truck', 'V1B_bat_LSB_perkWh', 'V1B_bat_prmLIBs_perkWh'], 
                                               years = [year_to_get_inP_Yof_bat_repl2], SSP = ssp_to_get_inP) 

    all_app_bat = []  
    for bat_x in ["LIB_NMC622",  'LIB_LFP',  'LIB_LTO', 'LIB_NCA', "LSB"]:
        bat_avail = data2.sel(Tech = bat_x, powertrain = "BEV", Module = "V1B", v_type = "garbage_truck",  
                              SSP = ssp_to_get_inP, v_year = year_to_get_inP_Yof_bat_repl2)["tech_avail"].values
        if np.isnan(bat_avail).all(): 
            pass
        else: 
            all_app_bat.append(bat_x)
    print(f" @@@@@@ for SSP {ssp_to_get_inP} year {year_to_get_inP}, available {all_app_bat}")

    for b_tech in all_app_bat :
        if b_tech == "LIB_NMC622":    #mapping names in data2, to be updated 
            b_tech = "NMC622"
        elif b_tech == "LIB_LFP":
            b_tech = "LFP"
        elif b_tech == "LIB_LTO":
            b_tech = "LTO"
        elif b_tech == "LIB_NCA": 
            b_tech = "NCA"
        
        df= agb.multiLCAAlgebric(models=[
                (bat_final_whSwitchbat, 1)],  #act_test
                methods = methods_touse,   
                gtruck_BEV_battery_tech = b_tech ,
                **yearlyParams
            )

        print(f"end of running LCA calc for SSP {ssp_to_get_inP} and year {year_to_get_inP} using battery tech {b_tech} ")
        display(df)
        
        # prepare for final DF: 
        GWP0 = df.iloc[:,0].values[0]
        GWP1 = df.iloc[:,1].values[0]
        GWP2 = df.iloc[:,2].values[0]
        GWP0_list.append(GWP0)
        GWP1_list.append(GWP1)
        GWP2_list.append(GWP2)
        SSP_list.append(ssp_to_get_inP) 
        Year_list.append(year_to_get_inP)
        tech_list.append(b_tech)
        

 @@@@@@ for SSP ssp119 year 2030, available ['LIB_NMC622', 'LIB_LFP', 'LIB_LTO', 'LIB_NCA', 'LSB']
end of running LCA calc for SSP ssp119 and year 2030 using battery tech NMC622 


,climate change no LT - global warming potential (GWP100) no LT[kg CO2-Eq],"climate change - GWP 100a, incl. H and bio CO2[kg CO2-Eq]",climate change - pGWP100[kg CO2-Eq]
V1B_bat_g-truck BEV_wh_switch_bat_wh__remind_SSP1-PkBudg500_2030,61914.716088,63684.844495,62289.142394


end of running LCA calc for SSP ssp119 and year 2030 using battery tech LFP 


,climate change no LT - global warming potential (GWP100) no LT[kg CO2-Eq],"climate change - GWP 100a, incl. H and bio CO2[kg CO2-Eq]",climate change - pGWP100[kg CO2-Eq]
V1B_bat_g-truck BEV_wh_switch_bat_wh__remind_SSP1-PkBudg500_2030,53841.295559,54347.331486,53276.933478


end of running LCA calc for SSP ssp119 and year 2030 using battery tech LTO 


...

,climate change no LT - global warming potential (GWP100) no LT[kg CO2-Eq],"climate change - GWP 100a, incl. H and bio CO2[kg CO2-Eq]",climate change - pGWP100[kg CO2-Eq]
V1B_bat_g-truck BEV_wh_switch_bat_wh__remind_SSP5-Base_2050,63827.521667,64427.414718,63358.792078


end of running LCA calc for SSP ssp585 and year 2050 using battery tech LTO 


,climate change no LT - global warming potential (GWP100) no LT[kg CO2-Eq],"climate change - GWP 100a, incl. H and bio CO2[kg CO2-Eq]",climate change - pGWP100[kg CO2-Eq]
V1B_bat_g-truck BEV_wh_switch_bat_wh__remind_SSP5-Base_2050,93510.187177,96278.34638,94833.261783


end of running LCA calc for SSP ssp585 and year 2050 using battery tech NCA 


,climate change no LT - global warming potential (GWP100) no LT[kg CO2-Eq],"climate change - GWP 100a, incl. H and bio CO2[kg CO2-Eq]",climate change - pGWP100[kg CO2-Eq]
V1B_bat_g-truck BEV_wh_switch_bat_wh__remind_SSP5-Base_2050,67058.348942,71102.12503,69983.244525


### prepare final DF and DS

In [20]:
d1 = get_plcia_wh_V1AV1B_BEV(SSP_list, Year_list, tech_list, GWP0_list, GWP1_list, GWP2_list, v_module = "V1B_repl")[0]
d2 = get_plcia_wh_V1AV1B_BEV(SSP_list, Year_list, tech_list, GWP0_list, GWP1_list, GWP2_list, v_module = "V1B_repl")[1]

In [21]:
bat_ds = get_plcia_ds(d2)
bat_ds

<xarray.Dataset>
Dimensions:          (SSP: 3, v_module: 1, v_year: 3, tech_list: 5,
                      Impact_category: 3)
Coordinates:
  * SSP              (SSP) object 'ssp119' 'ssp245' 'ssp585'
  * v_module         (v_module) object 'V1B_repl'
  * v_year           (v_year) int64 2030 2040 2050
  * tech_list        (tech_list) object 'LFP' 'LSB' 'LTO' 'NCA' 'NMC622'
  * Impact_category  (Impact_category) object 'IPCC2021_default' ... 'premise...
Data variables:
    value            (SSP, v_module, v_year, tech_list, Impact_category) float64 ...

In [22]:
bat_ds.to_netcdf('pGWP_module_output/V1B_batrepl_BEVGT_pGWP100.nc')

In [23]:
d1.to_excel('pGWP_module_output/excel_output/V1B_batrepl_BEVGT_pGWP100_NMConly.xlsx' )
d2.to_excel('pGWP_module_output/excel_output/V1B_batrepl_BEVGT_pGWP100_alltech.xlsx' )